In [19]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Notebook dataset folder-এর ভেতরে বা তার parent folder থেকে চালানো যাবে
CURRENT_DIR = Path.cwd()

ROOT = (
    CURRENT_DIR
    if (CURRENT_DIR / "train_transcripts").exists()
    else CURRENT_DIR / "Trace-The-Race-Dataset"
)

TRANSCRIPT_DIR = ROOT / "train_transcripts"

OUTPUT_DIR = ROOT / "outputs" / "01_transcript_merge"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "train_transcripts_by_session.parquet"

print("Dataset root :", ROOT)
print("Transcript dir:", TRANSCRIPT_DIR)
print("Output file   :", OUTPUT_FILE)

Dataset root : c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset
Transcript dir: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts
Output file   : c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\01_transcript_merge\train_transcripts_by_session.parquet


In [2]:
transcript_files = sorted(TRANSCRIPT_DIR.glob("*.csv"))

batch_size = 500
batch_rows = []
writer = None

for file_number, file_path in enumerate(transcript_files, start=1):

    # একটি session transcript load
    df = pd.read_csv(
        file_path,
        usecols=[
            "session_id",
            "utterance_id",
            "role",
            "content",
            "timestamp",
        ],
    )

    # সঠিক conversation order
    df = df.sort_values(
        "utterance_id",
        kind="stable",
    ).reset_index(drop=True)

    # Text clean
    roles = (
        df["role"]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    contents = (
        df["content"]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Speaker-tagged conversation
    tagged_lines = "[" + roles + "] " + contents

    session_id = (
        str(df["session_id"].iloc[0])
        if len(df) > 0
        else file_path.stem
    )

    batch_rows.append(
        {
            "session_id": session_id,
            "source_file": file_path.name,

            # Complete conversation
            "transcript_text": "\n".join(tagged_lines.tolist()),

            # Speaker-specific text
            "student_text": "\n".join(
                contents[roles.eq("STUDENT")].tolist()
            ),
            "tutor_text": "\n".join(
                contents[roles.eq("TUTOR")].tolist()
            ),

            # Simple session information
            "total_turns": int(len(df)),
            "student_turns": int(roles.eq("STUDENT").sum()),
            "tutor_turns": int(roles.eq("TUTOR").sum()),
            "background_turns": int(roles.eq("BACKGROUND").sum()),
        }
    )

    # প্রতি 500 session পর Parquet-এ write
    if len(batch_rows) == batch_size or file_number == len(transcript_files):

        table = pa.Table.from_pylist(batch_rows)

        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT_FILE,
                table.schema,
                compression="zstd",
            )

        writer.write_table(table)
        batch_rows.clear()

        print(
            f"Processed: {file_number:,} / "
            f"{len(transcript_files):,} files"
        )

if writer is not None:
    writer.close()

print("\nCompleted")
print("Saved:", OUTPUT_FILE)
print(
    "File size:",
    f"{OUTPUT_FILE.stat().st_size / 1024**2:.2f} MB",
)

Processed: 500 / 22,821 files
Processed: 1,000 / 22,821 files
Processed: 1,500 / 22,821 files
Processed: 2,000 / 22,821 files
Processed: 2,500 / 22,821 files
Processed: 3,000 / 22,821 files
Processed: 3,500 / 22,821 files
Processed: 4,000 / 22,821 files
Processed: 4,500 / 22,821 files
Processed: 5,000 / 22,821 files
Processed: 5,500 / 22,821 files
Processed: 6,000 / 22,821 files
Processed: 6,500 / 22,821 files
Processed: 7,000 / 22,821 files
Processed: 7,500 / 22,821 files
Processed: 8,000 / 22,821 files
Processed: 8,500 / 22,821 files
Processed: 9,000 / 22,821 files
Processed: 9,500 / 22,821 files
Processed: 10,000 / 22,821 files
Processed: 10,500 / 22,821 files
Processed: 11,000 / 22,821 files
Processed: 11,500 / 22,821 files
Processed: 12,000 / 22,821 files
Processed: 12,500 / 22,821 files
Processed: 13,000 / 22,821 files
Processed: 13,500 / 22,821 files
Processed: 14,000 / 22,821 files
Processed: 14,500 / 22,821 files
Processed: 15,000 / 22,821 files
Processed: 15,500 / 22,821 file

In [3]:
parquet_file = pq.ParquetFile(OUTPUT_FILE)

print("Rows       :", parquet_file.metadata.num_rows)
print("Row groups :", parquet_file.metadata.num_row_groups)
print("Columns    :", parquet_file.schema.names)
print(
    "File size  :",
    f"{OUTPUT_FILE.stat().st_size / 1024**2:.2f} MB",
)

preview = (
    parquet_file
    .read_row_group(0)
    .to_pandas()
    .head(3)
)

display(
    preview[
        [
            "session_id",
            "total_turns",
            "student_turns",
            "tutor_turns",
        ]
    ]
)

Rows       : 22821
Row groups : 46
Columns    : ['session_id', 'source_file', 'transcript_text', 'student_text', 'tutor_text', 'total_turns', 'student_turns', 'tutor_turns', 'background_turns']
File size  : 265.02 MB


,session_id,total_turns,student_turns,tutor_turns
0,aaaedit,254,114,136
1,aaaptjd,360,178,165
2,aabkeov,281,136,137


## Group A — Session Size and Pace

This feature group measures the overall size, duration, and conversational
pace of each tutoring session.

### Features

1. `total_turns`  
   Total number of utterances in the session.

2. `total_words`  
   Total number of words spoken by the tutor, student, and background roles.

3. `session_duration_minutes`  
   Time difference between the first and last valid timestamps.

4. `turns_per_minute`  
   Average number of transcript turns per minute.

### Formulas

```text
session_duration_minutes =
(last valid timestamp − first valid timestamp) / 60

turns_per_minute =
total_turns / session_duration_minutes

## Group A — Session Size and Pace

This group measures the overall size, duration, and conversational pace
of each tutoring session.

### Features

1. `total_turns`  
   Total number of utterances in the session.

2. `total_words`  
   Total number of words across all transcript turns.

3. `session_duration_minutes`  
   Time difference between the first and last valid timestamps.

4. `turns_per_minute`  
   Average number of transcript turns per minute.

### Formulas

```text
session_duration_minutes =
(last valid timestamp − first valid timestamp) / 60

turns_per_minute =
total_turns / session_duration_minutes

In [5]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd


# =========================================================
# 1. FIND THE DATASET FOLDER
# =========================================================
CURRENT_DIR = Path.cwd().resolve()

ROOT_CANDIDATES = [
    CURRENT_DIR,
    CURRENT_DIR / "Trace-The-Race-Dataset",
    CURRENT_DIR.parent / "Trace-The-Race-Dataset",
]

ROOT = next(
    (
        path
        for path in ROOT_CANDIDATES
        if (path / "train_transcripts").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Trace-The-Race-Dataset/train_transcripts folder was not found."
    )

TRANSCRIPT_DIR = ROOT / "train_transcripts"

OUTPUT_DIR = ROOT / "outputs" / "02_feature_groups"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GROUP_A_FILE = (
    OUTPUT_DIR
    / "group_a_session_size_and_pace.parquet"
)

transcript_files = sorted(TRANSCRIPT_DIR.glob("*.csv"))

if not transcript_files:
    raise FileNotFoundError(
        f"No CSV files found inside: {TRANSCRIPT_DIR}"
    )

print("Dataset root      :", ROOT)
print("Transcript folder :", TRANSCRIPT_DIR)
print("Transcript files  :", f"{len(transcript_files):,}")


Dataset root      : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset
Transcript folder : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts
Transcript files  : 22,821


In [6]:
# =========================================================
# 2. PROCESS EACH TRANSCRIPT FILE
# =========================================================
required_columns = {"session_id", "utterance_id", "role", "content", "timestamp"}

session_rows = []
start_time = perf_counter()

for file_number, file_path in enumerate(transcript_files, start=1):

    df = pd.read_csv(
        file_path,
        usecols=lambda column: column in required_columns,
        dtype={"session_id": "string", "role": "string", "content": "string", "timestamp": "string"},
    )

    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(f"{file_path.name} is missing columns: {sorted(missing_columns)}")

    # Session ID
    valid_session_ids = df["session_id"].dropna().astype(str).str.strip()
    session_id = valid_session_ids.iloc[0] if len(valid_session_ids) > 0 else file_path.stem

    # Clean transcript content
    content = df["content"].fillna("").astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

    # Count words in every utterance
    word_counts = content.str.count(r"\b[\w']+\b").fillna(0)

    total_turns = int(len(df))
    total_words = int(word_counts.sum())

    # Convert timestamp to seconds
    timestamp_seconds = pd.to_timedelta(df["timestamp"], errors="coerce").dt.total_seconds().dropna()

    if len(timestamp_seconds) >= 2:
        duration_seconds = float(timestamp_seconds.max() - timestamp_seconds.min())
        duration_seconds = max(duration_seconds, 0.0)
    else:
        duration_seconds = 0.0

    session_duration_minutes = duration_seconds / 60.0
    turns_per_minute = total_turns / session_duration_minutes if session_duration_minutes > 0 else 0.0

    session_rows.append(
        {
            "session_id": session_id,
            "source_file": file_path.name,
            "total_turns": total_turns,
            "total_words": total_words,
            "session_duration_minutes": session_duration_minutes,
            "turns_per_minute": turns_per_minute,
        }
    )

    # Progress
    if file_number % 500 == 0 or file_number == len(transcript_files):
        elapsed = perf_counter() - start_time
        speed = file_number / elapsed if elapsed > 0 else 0
        remaining = len(transcript_files) - file_number
        eta_minutes = remaining / speed / 60 if speed > 0 else 0

        print(f"Processed {file_number:,}/{len(transcript_files):,} | ETA: {eta_minutes:.1f} min")


Processed 500/22,821 | ETA: 1.4 min
Processed 1,000/22,821 | ETA: 1.3 min
Processed 1,500/22,821 | ETA: 1.3 min
Processed 2,000/22,821 | ETA: 1.3 min
Processed 2,500/22,821 | ETA: 1.2 min
Processed 3,000/22,821 | ETA: 1.2 min
Processed 3,500/22,821 | ETA: 1.2 min
Processed 4,000/22,821 | ETA: 1.1 min
Processed 4,500/22,821 | ETA: 1.1 min
Processed 5,000/22,821 | ETA: 1.1 min
Processed 5,500/22,821 | ETA: 1.1 min
Processed 6,000/22,821 | ETA: 1.0 min
Processed 6,500/22,821 | ETA: 1.0 min
Processed 7,000/22,821 | ETA: 1.0 min
Processed 7,500/22,821 | ETA: 0.9 min
Processed 8,000/22,821 | ETA: 0.9 min
Processed 8,500/22,821 | ETA: 0.9 min
Processed 9,000/22,821 | ETA: 0.8 min
Processed 9,500/22,821 | ETA: 0.8 min
Processed 10,000/22,821 | ETA: 0.8 min
Processed 10,500/22,821 | ETA: 0.8 min
Processed 11,000/22,821 | ETA: 0.7 min
Processed 11,500/22,821 | ETA: 0.7 min
Processed 12,000/22,821 | ETA: 0.7 min
Processed 12,500/22,821 | ETA: 0.6 min
Processed 13,000/22,821 | ETA: 0.6 min
Process

In [7]:

# =========================================================
# 3. CREATE GROUP A TABLE
# =========================================================
group_a = pd.DataFrame(session_rows)

group_a["total_turns"] = group_a["total_turns"].astype("int32")
group_a["total_words"] = group_a["total_words"].astype("int32")

group_a["session_duration_minutes"] = (
    group_a["session_duration_minutes"]
    .replace([np.inf, -np.inf], 0.0)
    .fillna(0.0)
    .astype("float32")
)

group_a["turns_per_minute"] = (
    group_a["turns_per_minute"]
    .replace([np.inf, -np.inf], 0.0)
    .fillna(0.0)
    .astype("float32")
)



In [9]:

# =========================================================
# 4. VALIDATE
# =========================================================
if not group_a["session_id"].is_unique:
    duplicates = group_a.loc[group_a["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate session IDs found: {duplicates[:10]}")

feature_columns_a = ["total_turns", "total_words", "session_duration_minutes", "turns_per_minute"]

if group_a[feature_columns_a].isna().any().any():
    raise ValueError("Missing values remain in Group A features.")


# =========================================================
# 5. SAVE AS PARQUET
# =========================================================
group_a.to_parquet(GROUP_A_FILE, index=False, engine="pyarrow", compression="snappy")

elapsed_minutes = (perf_counter() - start_time) / 60

print("\nGroup A completed successfully.")
print("Shape      :", group_a.shape)
print("Time       :", f"{elapsed_minutes:.2f} minutes")
print("Saved file :", GROUP_A_FILE)
print("File size  :", f"{GROUP_A_FILE.stat().st_size / 1024**2:.2f} MB")

display(group_a.head(100))


Group A completed successfully.
Shape      : (22821, 6)
Time       : 5.94 minutes
Saved file : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\02_feature_groups\group_a_session_size_and_pace.parquet
File size  : 0.67 MB


,session_id,source_file,total_turns,total_words,session_duration_minutes,turns_per_minute
0,aaaedit,aaaedit.csv,254,3156,43.816666,5.796881
1,aaaptjd,aaaptjd.csv,360,6376,45.500000,7.912088
2,aabkeov,aabkeov.csv,281,3045,36.799999,7.635870
3,aacggvb,aacggvb.csv,235,4180,46.266666,5.079251
4,aadexbc,aadexbc.csv,104,2676,44.033333,2.361847
...,...,...,...,...,...,...
95,absjqop,absjqop.csv,234,4396,44.233334,5.290128
96,absmyud,absmyud.csv,278,4678,46.833332,5.935943
97,abtepuj,abtepuj.csv,103,3771,33.299999,3.093093
98,abtmcwd,abtmcwd.csv,339,3847,43.183334,7.850251


## Group B — Student and Tutor Participation

This group measures how much the student and tutor participate in each tutoring session.

### Features

5. `student_turns`  
   Total number of utterances produced by the student.

6. `tutor_turns`  
   Total number of utterances produced by the tutor.

7. `student_turn_ratio`  
   Proportion of all session turns produced by the student.

8. `student_word_ratio`  
   Proportion of all transcript words spoken by the student.

9. `avg_student_words_per_turn`  
   Average number of words in each student utterance.

10. `avg_tutor_words_per_turn`  
    Average number of words in each tutor utterance.

### Formulas

```text
student_turn_ratio = student_turns / total_turns

student_word_ratio = student_words / total_words

avg_student_words_per_turn = student_words / student_turns

avg_tutor_words_per_turn = tutor_words / tutor_turns

In [10]:
# =========================================================
# SECTION 2 — STUDENT AND TUTOR PARTICIPATION
# =========================================================
GROUP_B_FILE = OUTPUT_DIR / "group_b_student_tutor_participation.parquet"

section_b_rows = []
start_time_b = perf_counter()

for file_number, file_path in enumerate(transcript_files, start=1):

    df = pd.read_csv(file_path, usecols=["session_id", "role", "content"], dtype={"session_id": "string", "role": "string", "content": "string"})

    # Session ID
    valid_session_ids = df["session_id"].dropna().astype(str).str.strip()
    session_id = valid_session_ids.iloc[0] if len(valid_session_ids) > 0 else file_path.stem

    # Clean role and content
    role = df["role"].fillna("UNKNOWN").astype("string").str.strip().str.upper()
    content = df["content"].fillna("").astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

    # Create role masks
    student_mask = role.eq("STUDENT")
    tutor_mask = role.eq("TUTOR")

    # Count words in each turn
    word_counts = content.str.count(r"\b[\w']+\b").fillna(0)

    # Raw counts
    total_turns = int(len(df))
    total_words = int(word_counts.sum())
    student_turns = int(student_mask.sum())
    tutor_turns = int(tutor_mask.sum())
    student_words = int(word_counts[student_mask].sum())
    tutor_words = int(word_counts[tutor_mask].sum())

    # Safe ratio calculations
    student_turn_ratio = student_turns / total_turns if total_turns > 0 else 0.0
    student_word_ratio = student_words / total_words if total_words > 0 else 0.0
    avg_student_words_per_turn = student_words / student_turns if student_turns > 0 else 0.0
    avg_tutor_words_per_turn = tutor_words / tutor_turns if tutor_turns > 0 else 0.0

    section_b_rows.append({"session_id": session_id, "student_turns": student_turns, "tutor_turns": tutor_turns, "student_turn_ratio": student_turn_ratio, "student_word_ratio": student_word_ratio, "avg_student_words_per_turn": avg_student_words_per_turn, "avg_tutor_words_per_turn": avg_tutor_words_per_turn})

    # Progress
    if file_number % 500 == 0 or file_number == len(transcript_files):
        elapsed = perf_counter() - start_time_b
        speed = file_number / elapsed if elapsed > 0 else 0
        remaining = len(transcript_files) - file_number
        eta_minutes = remaining / speed / 60 if speed > 0 else 0

        print(f"Processed {file_number:,}/{len(transcript_files):,} | ETA: {eta_minutes:.1f} min")


Processed 500/22,821 | ETA: 1.4 min
Processed 1,000/22,821 | ETA: 1.3 min
Processed 1,500/22,821 | ETA: 1.3 min
Processed 2,000/22,821 | ETA: 1.3 min
Processed 2,500/22,821 | ETA: 1.2 min
Processed 3,000/22,821 | ETA: 1.2 min
Processed 3,500/22,821 | ETA: 1.2 min
Processed 4,000/22,821 | ETA: 1.2 min
Processed 4,500/22,821 | ETA: 1.1 min
Processed 5,000/22,821 | ETA: 1.1 min
Processed 5,500/22,821 | ETA: 1.1 min
Processed 6,000/22,821 | ETA: 1.1 min
Processed 6,500/22,821 | ETA: 1.0 min
Processed 7,000/22,821 | ETA: 1.0 min
Processed 7,500/22,821 | ETA: 1.0 min
Processed 8,000/22,821 | ETA: 1.0 min
Processed 8,500/22,821 | ETA: 0.9 min
Processed 9,000/22,821 | ETA: 0.9 min
Processed 9,500/22,821 | ETA: 0.8 min
Processed 10,000/22,821 | ETA: 0.8 min
Processed 10,500/22,821 | ETA: 0.8 min
Processed 11,000/22,821 | ETA: 0.7 min
Processed 11,500/22,821 | ETA: 0.7 min
Processed 12,000/22,821 | ETA: 0.7 min
Processed 12,500/22,821 | ETA: 0.6 min
Processed 13,000/22,821 | ETA: 0.6 min
Process

In [11]:

# =========================================================
# CREATE GROUP B TABLE
# =========================================================
group_b = pd.DataFrame(section_b_rows)

group_b["student_turns"] = group_b["student_turns"].astype("int32")
group_b["tutor_turns"] = group_b["tutor_turns"].astype("int32")

ratio_columns_b = ["student_turn_ratio", "student_word_ratio", "avg_student_words_per_turn", "avg_tutor_words_per_turn"]

group_b[ratio_columns_b] = group_b[ratio_columns_b].replace([np.inf, -np.inf], 0.0).fillna(0.0).astype("float32")


In [13]:
# =========================================================
# VALIDATE GROUP B
# =========================================================
if not group_b["session_id"].is_unique:
    duplicates = group_b.loc[group_b["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate session IDs found: {duplicates[:10]}")

feature_columns_b = ["student_turns", "tutor_turns", "student_turn_ratio", "student_word_ratio", "avg_student_words_per_turn", "avg_tutor_words_per_turn"]

if group_b[feature_columns_b].isna().any().any():
    raise ValueError("Missing values remain in Group B features.")


# =========================================================
# SAVE GROUP B
# =========================================================
group_b.to_parquet(GROUP_B_FILE, index=False, engine="pyarrow", compression="snappy")


# =========================================================
# COMBINE GROUP A AND GROUP B
# =========================================================
session_features = group_a.merge(group_b, on="session_id", how="left", validate="one_to_one")

elapsed_minutes_b = (perf_counter() - start_time_b) / 60

print("\nGroup B completed successfully.")
print("Group B shape       :", group_b.shape)
print("Combined table shape:", session_features.shape)
print("Time                :", f"{elapsed_minutes_b:.2f} minutes")
print("Saved file          :", GROUP_B_FILE)
print("File size           :", f"{GROUP_B_FILE.stat().st_size / 1024**2:.2f} MB")

display(session_features.head(100))


Group B completed successfully.
Group B shape       : (22821, 7)
Combined table shape: (22821, 12)
Time                : 4.20 minutes
Saved file          : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\02_feature_groups\group_b_student_tutor_participation.parquet
File size           : 0.69 MB


,session_id,source_file,total_turns,total_words,session_duration_minutes,turns_per_minute,student_turns,tutor_turns,student_turn_ratio,student_word_ratio,avg_student_words_per_turn,avg_tutor_words_per_turn
0,aaaedit,aaaedit.csv,254,3156,43.816666,5.796881,114,136,0.448819,0.392269,10.859649,14.000000
1,aaaptjd,aaaptjd.csv,360,6376,45.500000,7.912088,178,165,0.494444,0.269134,9.640450,26.642424
2,aabkeov,aabkeov.csv,281,3045,36.799999,7.635870,136,137,0.483986,0.359606,8.051471,13.620438
3,aacggvb,aacggvb.csv,235,4180,46.266666,5.079251,112,118,0.476596,0.282775,10.553572,24.983051
4,aadexbc,aadexbc.csv,104,2676,44.033333,2.361847,20,81,0.192308,0.057175,7.650000,31.037037
...,...,...,...,...,...,...,...,...,...,...,...,...
95,absjqop,absjqop.csv,234,4396,44.233334,5.290128,98,127,0.418803,0.238626,10.704082,25.850393
96,absmyud,absmyud.csv,278,4678,46.833332,5.935943,122,145,0.438849,0.306755,11.762295,21.379311
97,abtepuj,abtepuj.csv,103,3771,33.299999,3.093093,30,69,0.291262,0.017237,2.166667,52.304348
98,abtmcwd,abtmcwd.csv,339,3847,43.183334,7.850251,144,185,0.424779,0.216272,5.777778,15.805406


## Group C — Student Response Style

This group measures the visible structure and style of student responses.

### Features

11. `student_short_turn_ratio`  
    Proportion of student turns containing three words or fewer.

12. `student_long_turn_ratio`  
    Proportion of student turns containing ten words or more.

13. `student_numeric_turn_ratio`  
    Proportion of student turns containing at least one numeric digit.

14. `student_question_ratio`  
    Proportion of student turns containing a question mark.

### Formulas

```text
student_short_turn_ratio =
student turns with <= 3 words / student_turns

student_long_turn_ratio =
student turns with >= 10 words / student_turns

student_numeric_turn_ratio =
student turns containing a digit / student_turns

student_question_ratio =
student turns containing "?" / student_turns

In [14]:

# =========================================================
# SECTION 3 — STUDENT RESPONSE STYLE
# =========================================================
GROUP_C_FILE = OUTPUT_DIR / "group_c_student_response_style.parquet"

section_c_rows = []
start_time_c = perf_counter()

for file_number, file_path in enumerate(transcript_files, start=1):

    df = pd.read_csv(file_path, usecols=["session_id", "role", "content"], dtype={"session_id": "string", "role": "string", "content": "string"})

    # Session ID
    valid_session_ids = df["session_id"].dropna().astype(str).str.strip()
    session_id = valid_session_ids.iloc[0] if len(valid_session_ids) > 0 else file_path.stem

    # Clean role and content
    role = df["role"].fillna("UNKNOWN").astype("string").str.strip().str.upper()
    content = df["content"].fillna("").astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

    # Keep only student turns
    student_mask = role.eq("STUDENT")
    student_content = content[student_mask]
    student_turns = int(student_mask.sum())

    # Student turn properties
    student_word_counts = student_content.str.count(r"\b[\w']+\b").fillna(0)
    student_short_turns = int(student_word_counts.le(3).sum())
    student_long_turns = int(student_word_counts.ge(10).sum())
    student_numeric_turns = int(student_content.str.contains(r"\d", regex=True, na=False).sum())
    student_question_turns = int(student_content.str.contains("?", regex=False, na=False).sum())

    # Safe ratio calculations
    student_short_turn_ratio = student_short_turns / student_turns if student_turns > 0 else 0.0
    student_long_turn_ratio = student_long_turns / student_turns if student_turns > 0 else 0.0
    student_numeric_turn_ratio = student_numeric_turns / student_turns if student_turns > 0 else 0.0
    student_question_ratio = student_question_turns / student_turns if student_turns > 0 else 0.0

    section_c_rows.append({"session_id": session_id, "student_short_turn_ratio": student_short_turn_ratio, "student_long_turn_ratio": student_long_turn_ratio, "student_numeric_turn_ratio": student_numeric_turn_ratio, "student_question_ratio": student_question_ratio})

    # Progress
    if file_number % 500 == 0 or file_number == len(transcript_files):
        elapsed = perf_counter() - start_time_c
        speed = file_number / elapsed if elapsed > 0 else 0
        remaining = len(transcript_files) - file_number
        eta_minutes = remaining / speed / 60 if speed > 0 else 0

        print(f"Processed {file_number:,}/{len(transcript_files):,} | ETA: {eta_minutes:.1f} min")


# =========================================================
# CREATE GROUP C TABLE
# =========================================================
group_c = pd.DataFrame(section_c_rows)

feature_columns_c = ["student_short_turn_ratio", "student_long_turn_ratio", "student_numeric_turn_ratio", "student_question_ratio"]

group_c[feature_columns_c] = group_c[feature_columns_c].replace([np.inf, -np.inf], 0.0).fillna(0.0).astype("float32")


# =========================================================
# VALIDATE GROUP C
# =========================================================
if not group_c["session_id"].is_unique:
    duplicates = group_c.loc[group_c["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate session IDs found: {duplicates[:10]}")

if group_c[feature_columns_c].isna().any().any():
    raise ValueError("Missing values remain in Group C features.")

if not group_c[feature_columns_c].apply(lambda column: column.between(0, 1).all()).all():
    raise ValueError("One or more Group C ratios are outside the 0–1 range.")


# =========================================================
# SAVE GROUP C
# =========================================================
group_c.to_parquet(GROUP_C_FILE, index=False, engine="pyarrow", compression="snappy")


# =========================================================
# COMBINE GROUP A, GROUP B AND GROUP C
# =========================================================
session_features = session_features.merge(group_c, on="session_id", how="left", validate="one_to_one")

elapsed_minutes_c = (perf_counter() - start_time_c) / 60

print("\nGroup C completed successfully.")
print("Group C shape       :", group_c.shape)
print("Combined table shape:", session_features.shape)
print("Time                :", f"{elapsed_minutes_c:.2f} minutes")
print("Saved file          :", GROUP_C_FILE)
print("File size           :", f"{GROUP_C_FILE.stat().st_size / 1024**2:.2f} MB")

display(session_features.head())

Processed 500/22,821 | ETA: 1.2 min
Processed 1,000/22,821 | ETA: 1.2 min
Processed 1,500/22,821 | ETA: 1.2 min
Processed 2,000/22,821 | ETA: 1.2 min
Processed 2,500/22,821 | ETA: 1.1 min
Processed 3,000/22,821 | ETA: 1.1 min
Processed 3,500/22,821 | ETA: 1.1 min
Processed 4,000/22,821 | ETA: 1.0 min
Processed 4,500/22,821 | ETA: 1.0 min
Processed 5,000/22,821 | ETA: 1.0 min
Processed 5,500/22,821 | ETA: 0.9 min
Processed 6,000/22,821 | ETA: 0.9 min
Processed 6,500/22,821 | ETA: 0.9 min
Processed 7,000/22,821 | ETA: 0.9 min
Processed 7,500/22,821 | ETA: 0.8 min
Processed 8,000/22,821 | ETA: 0.8 min
Processed 8,500/22,821 | ETA: 0.8 min
Processed 9,000/22,821 | ETA: 0.8 min
Processed 9,500/22,821 | ETA: 0.7 min
Processed 10,000/22,821 | ETA: 0.7 min
Processed 10,500/22,821 | ETA: 0.7 min
Processed 11,000/22,821 | ETA: 0.6 min
Processed 11,500/22,821 | ETA: 0.6 min
Processed 12,000/22,821 | ETA: 0.6 min
Processed 12,500/22,821 | ETA: 0.6 min
Processed 13,000/22,821 | ETA: 0.5 min
Process

,session_id,source_file,total_turns,total_words,session_duration_minutes,turns_per_minute,student_turns,tutor_turns,student_turn_ratio,student_word_ratio,avg_student_words_per_turn,avg_tutor_words_per_turn,student_short_turn_ratio,student_long_turn_ratio,student_numeric_turn_ratio,student_question_ratio
0,aaaedit,aaaedit.csv,254,3156,43.816666,5.796881,114,136,0.448819,0.392269,10.859649,14.000000,0.429825,0.333333,0.236842,0.333333
1,aaaptjd,aaaptjd.csv,360,6376,45.500000,7.912088,178,165,0.494444,0.269134,9.640450,26.642424,0.471910,0.252809,0.516854,0.202247
2,aabkeov,aabkeov.csv,281,3045,36.799999,7.635870,136,137,0.483986,0.359606,8.051471,13.620438,0.529412,0.213235,0.205882,0.257353
3,aacggvb,aacggvb.csv,235,4180,46.266666,5.079251,112,118,0.476596,0.282775,10.553572,24.983051,0.553571,0.321429,0.223214,0.160714
4,aadexbc,aadexbc.csv,104,2676,44.033333,2.361847,20,81,0.192308,0.057175,7.650000,31.037037,0.600000,0.200000,0.150000,0.250000


## Group D — Tutor Questioning and Response Flow

This group measures how frequently the tutor asks questions and whether
those questions receive an immediate student response.

### Features

15. `tutor_question_ratio`  
    Proportion of tutor turns containing a question mark.

16. `student_response_after_tutor_question_ratio`  
    Proportion of tutor questions whose immediately following turn belongs
    to the student.

### Formulas

```text
tutor_question_ratio =
tutor question turns / tutor_turns

student_response_after_tutor_question_ratio =
tutor questions immediately followed by a student turn
/
tutor question turns

In [15]:
# =========================================================
# SECTION 4 — TUTOR QUESTIONING AND RESPONSE FLOW
# =========================================================
GROUP_D_FILE = OUTPUT_DIR / "group_d_tutor_question_response_flow.parquet"

section_d_rows = []
start_time_d = perf_counter()

for file_number, file_path in enumerate(transcript_files, start=1):

    df = pd.read_csv(file_path, usecols=["session_id", "utterance_id", "role", "content"], dtype={"session_id": "string", "role": "string", "content": "string"})

    valid_session_ids = df["session_id"].dropna().astype(str).str.strip()
    session_id = valid_session_ids.iloc[0] if len(valid_session_ids) > 0 else file_path.stem

    df["_order"] = pd.to_numeric(df["utterance_id"], errors="coerce")
    df = df.sort_values("_order", kind="stable", na_position="last").reset_index(drop=True)

    role = df["role"].fillna("UNKNOWN").astype("string").str.strip().str.upper()
    content = df["content"].fillna("").astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

    tutor_mask = role.eq("TUTOR")
    tutor_question_mask = tutor_mask & content.str.contains("?", regex=False, na=False)

    tutor_turns = int(tutor_mask.sum())
    tutor_question_turns = int(tutor_question_mask.sum())

    next_role = role.shift(-1)
    student_responses_after_questions = int((tutor_question_mask & next_role.eq("STUDENT")).sum())

    tutor_question_ratio = tutor_question_turns / tutor_turns if tutor_turns > 0 else 0.0
    student_response_after_tutor_question_ratio = student_responses_after_questions / tutor_question_turns if tutor_question_turns > 0 else 0.0

    section_d_rows.append({"session_id": session_id, "tutor_question_ratio": tutor_question_ratio, "student_response_after_tutor_question_ratio": student_response_after_tutor_question_ratio})

    if file_number % 500 == 0 or file_number == len(transcript_files):
        elapsed = perf_counter() - start_time_d
        speed = file_number / elapsed if elapsed > 0 else 0
        remaining = len(transcript_files) - file_number
        eta_minutes = remaining / speed / 60 if speed > 0 else 0

        print(f"Processed {file_number:,}/{len(transcript_files):,} | ETA: {eta_minutes:.1f} min")


# =========================================================
# CREATE AND VALIDATE GROUP D TABLE
# =========================================================
group_d = pd.DataFrame(section_d_rows)

feature_columns_d = ["tutor_question_ratio", "student_response_after_tutor_question_ratio"]

group_d[feature_columns_d] = group_d[feature_columns_d].replace([np.inf, -np.inf], 0.0).fillna(0.0).astype("float32")

if not group_d["session_id"].is_unique:
    duplicates = group_d.loc[group_d["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate session IDs found: {duplicates[:10]}")

if group_d[feature_columns_d].isna().any().any():
    raise ValueError("Missing values remain in Group D features.")

if not group_d[feature_columns_d].apply(lambda column: column.between(0, 1).all()).all():
    raise ValueError("One or more Group D ratios are outside the 0–1 range.")


# =========================================================
# SAVE AND COMBINE
# =========================================================
group_d.to_parquet(GROUP_D_FILE, index=False, engine="pyarrow", compression="snappy")

session_features = session_features.merge(group_d, on="session_id", how="left", validate="one_to_one")

elapsed_minutes_d = (perf_counter() - start_time_d) / 60

print("\nGroup D completed successfully.")
print("Group D shape       :", group_d.shape)
print("Combined table shape:", session_features.shape)
print("Time                :", f"{elapsed_minutes_d:.2f} minutes")
print("Saved file          :", GROUP_D_FILE)

display(session_features.head())

Processed 500/22,821 | ETA: 1.2 min
Processed 1,000/22,821 | ETA: 1.2 min
Processed 1,500/22,821 | ETA: 1.2 min
Processed 2,000/22,821 | ETA: 1.2 min
Processed 2,500/22,821 | ETA: 1.2 min
Processed 3,000/22,821 | ETA: 1.2 min
Processed 3,500/22,821 | ETA: 1.2 min
Processed 4,000/22,821 | ETA: 1.2 min
Processed 4,500/22,821 | ETA: 1.1 min
Processed 5,000/22,821 | ETA: 1.1 min
Processed 5,500/22,821 | ETA: 1.1 min
Processed 6,000/22,821 | ETA: 1.0 min
Processed 6,500/22,821 | ETA: 1.0 min
Processed 7,000/22,821 | ETA: 0.9 min
Processed 7,500/22,821 | ETA: 0.9 min
Processed 8,000/22,821 | ETA: 0.9 min
Processed 8,500/22,821 | ETA: 0.8 min
Processed 9,000/22,821 | ETA: 0.8 min
Processed 9,500/22,821 | ETA: 0.8 min
Processed 10,000/22,821 | ETA: 0.8 min
Processed 10,500/22,821 | ETA: 0.7 min
Processed 11,000/22,821 | ETA: 0.7 min
Processed 11,500/22,821 | ETA: 0.7 min
Processed 12,000/22,821 | ETA: 0.6 min
Processed 12,500/22,821 | ETA: 0.6 min
Processed 13,000/22,821 | ETA: 0.6 min
Process

,session_id,source_file,total_turns,total_words,session_duration_minutes,turns_per_minute,student_turns,tutor_turns,student_turn_ratio,student_word_ratio,avg_student_words_per_turn,avg_tutor_words_per_turn,student_short_turn_ratio,student_long_turn_ratio,student_numeric_turn_ratio,student_question_ratio,tutor_question_ratio,student_response_after_tutor_question_ratio
0,aaaedit,aaaedit.csv,254,3156,43.816666,5.796881,114,136,0.448819,0.392269,10.859649,14.000000,0.429825,0.333333,0.236842,0.333333,0.507353,0.753623
1,aaaptjd,aaaptjd.csv,360,6376,45.500000,7.912088,178,165,0.494444,0.269134,9.640450,26.642424,0.471910,0.252809,0.516854,0.202247,0.721212,0.865546
2,aabkeov,aabkeov.csv,281,3045,36.799999,7.635870,136,137,0.483986,0.359606,8.051471,13.620438,0.529412,0.213235,0.205882,0.257353,0.583942,0.875000
3,aacggvb,aacggvb.csv,235,4180,46.266666,5.079251,112,118,0.476596,0.282775,10.553572,24.983051,0.553571,0.321429,0.223214,0.160714,0.627119,0.851351
4,aadexbc,aadexbc.csv,104,2676,44.033333,2.361847,20,81,0.192308,0.057175,7.650000,31.037037,0.600000,0.200000,0.150000,0.250000,0.777778,0.253968


## Group E — Conversation Dynamics

This group measures how the conversation moves between the tutor and
the student.

### Features

17. `speaker_switch_rate`  
    Frequency of speaker changes between tutor and student turns.

18. `longest_tutor_streak_ratio`  
    Longest consecutive tutor-turn streak divided by total tutor turns.

19. `longest_student_streak_ratio`  
    Longest consecutive student-turn streak divided by total student turns.

### Formulas

```text
speaker_switch_rate =
tutor-student speaker changes
/
number of possible dialogue transitions

longest_tutor_streak_ratio =
longest consecutive tutor streak / tutor_turns

longest_student_streak_ratio =
longest consecutive student streak / student_turns

In [16]:
# =========================================================
# SECTION 5 — CONVERSATION DYNAMICS
# =========================================================
GROUP_E_FILE = OUTPUT_DIR / "group_e_conversation_dynamics.parquet"


def find_longest_streak(role_sequence, target_role):
    longest = 0
    current = 0

    for current_role in role_sequence:
        if current_role == target_role:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest


section_e_rows = []
start_time_e = perf_counter()

for file_number, file_path in enumerate(transcript_files, start=1):

    df = pd.read_csv(file_path, usecols=["session_id", "utterance_id", "role"], dtype={"session_id": "string", "role": "string"})

    valid_session_ids = df["session_id"].dropna().astype(str).str.strip()
    session_id = valid_session_ids.iloc[0] if len(valid_session_ids) > 0 else file_path.stem

    df["_order"] = pd.to_numeric(df["utterance_id"], errors="coerce")
    df = df.sort_values("_order", kind="stable", na_position="last").reset_index(drop=True)

    role = df["role"].fillna("UNKNOWN").astype("string").str.strip().str.upper()
    dialogue_roles = role[role.isin(["TUTOR", "STUDENT"])].tolist()

    dialogue_turns = len(dialogue_roles)
    tutor_turns = dialogue_roles.count("TUTOR")
    student_turns = dialogue_roles.count("STUDENT")

    speaker_switches = sum(previous_role != current_role for previous_role, current_role in zip(dialogue_roles[:-1], dialogue_roles[1:]))

    speaker_switch_rate = speaker_switches / (dialogue_turns - 1) if dialogue_turns > 1 else 0.0

    longest_tutor_streak = find_longest_streak(dialogue_roles, "TUTOR")
    longest_student_streak = find_longest_streak(dialogue_roles, "STUDENT")

    longest_tutor_streak_ratio = longest_tutor_streak / tutor_turns if tutor_turns > 0 else 0.0
    longest_student_streak_ratio = longest_student_streak / student_turns if student_turns > 0 else 0.0

    section_e_rows.append({"session_id": session_id, "speaker_switch_rate": speaker_switch_rate, "longest_tutor_streak_ratio": longest_tutor_streak_ratio, "longest_student_streak_ratio": longest_student_streak_ratio})

    if file_number % 500 == 0 or file_number == len(transcript_files):
        elapsed = perf_counter() - start_time_e
        speed = file_number / elapsed if elapsed > 0 else 0
        remaining = len(transcript_files) - file_number
        eta_minutes = remaining / speed / 60 if speed > 0 else 0

        print(f"Processed {file_number:,}/{len(transcript_files):,} | ETA: {eta_minutes:.1f} min")


# =========================================================
# CREATE AND VALIDATE GROUP E TABLE
# =========================================================
group_e = pd.DataFrame(section_e_rows)

feature_columns_e = ["speaker_switch_rate", "longest_tutor_streak_ratio", "longest_student_streak_ratio"]

group_e[feature_columns_e] = group_e[feature_columns_e].replace([np.inf, -np.inf], 0.0).fillna(0.0).astype("float32")

if not group_e["session_id"].is_unique:
    duplicates = group_e.loc[group_e["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate session IDs found: {duplicates[:10]}")

if group_e[feature_columns_e].isna().any().any():
    raise ValueError("Missing values remain in Group E features.")

if not group_e[feature_columns_e].apply(lambda column: column.between(0, 1).all()).all():
    raise ValueError("One or more Group E ratios are outside the 0–1 range.")


# =========================================================
# SAVE AND COMBINE
# =========================================================
group_e.to_parquet(GROUP_E_FILE, index=False, engine="pyarrow", compression="snappy")

session_features = session_features.merge(group_e, on="session_id", how="left", validate="one_to_one")

elapsed_minutes_e = (perf_counter() - start_time_e) / 60

print("\nGroup E completed successfully.")
print("Group E shape       :", group_e.shape)
print("Combined table shape:", session_features.shape)
print("Time                :", f"{elapsed_minutes_e:.2f} minutes")
print("Saved file          :", GROUP_E_FILE)

display(session_features.head())

Processed 500/22,821 | ETA: 0.7 min
Processed 1,000/22,821 | ETA: 0.7 min
Processed 1,500/22,821 | ETA: 0.7 min
Processed 2,000/22,821 | ETA: 0.7 min
Processed 2,500/22,821 | ETA: 0.7 min
Processed 3,000/22,821 | ETA: 0.7 min
Processed 3,500/22,821 | ETA: 0.6 min
Processed 4,000/22,821 | ETA: 0.6 min
Processed 4,500/22,821 | ETA: 0.6 min
Processed 5,000/22,821 | ETA: 0.6 min
Processed 5,500/22,821 | ETA: 0.6 min
Processed 6,000/22,821 | ETA: 0.6 min
Processed 6,500/22,821 | ETA: 0.5 min
Processed 7,000/22,821 | ETA: 0.5 min
Processed 7,500/22,821 | ETA: 0.5 min
Processed 8,000/22,821 | ETA: 0.5 min
Processed 8,500/22,821 | ETA: 0.5 min
Processed 9,000/22,821 | ETA: 0.5 min
Processed 9,500/22,821 | ETA: 0.4 min
Processed 10,000/22,821 | ETA: 0.4 min
Processed 10,500/22,821 | ETA: 0.4 min
Processed 11,000/22,821 | ETA: 0.4 min
Processed 11,500/22,821 | ETA: 0.4 min
Processed 12,000/22,821 | ETA: 0.4 min
Processed 12,500/22,821 | ETA: 0.3 min
Processed 13,000/22,821 | ETA: 0.3 min
Process

,session_id,source_file,total_turns,total_words,session_duration_minutes,turns_per_minute,student_turns,tutor_turns,student_turn_ratio,student_word_ratio,...,avg_tutor_words_per_turn,student_short_turn_ratio,student_long_turn_ratio,student_numeric_turn_ratio,student_question_ratio,tutor_question_ratio,student_response_after_tutor_question_ratio,speaker_switch_rate,longest_tutor_streak_ratio,longest_student_streak_ratio
0,aaaedit,aaaedit.csv,254,3156,43.816666,5.796881,114,136,0.448819,0.392269,...,14.000000,0.429825,0.333333,0.236842,0.333333,0.507353,0.753623,0.779116,0.044118,0.035088
1,aaaptjd,aaaptjd.csv,360,6376,45.500000,7.912088,178,165,0.494444,0.269134,...,26.642424,0.471910,0.252809,0.516854,0.202247,0.721212,0.865546,0.862573,0.024242,0.022472
2,aabkeov,aabkeov.csv,281,3045,36.799999,7.635870,136,137,0.483986,0.359606,...,13.620438,0.529412,0.213235,0.205882,0.257353,0.583942,0.875000,0.849265,0.021898,0.029412
3,aacggvb,aacggvb.csv,235,4180,46.266666,5.079251,112,118,0.476596,0.282775,...,24.983051,0.553571,0.321429,0.223214,0.160714,0.627119,0.851351,0.816594,0.025424,0.026786
4,aadexbc,aadexbc.csv,104,2676,44.033333,2.361847,20,81,0.192308,0.057175,...,31.037037,0.600000,0.200000,0.150000,0.250000,0.777778,0.253968,0.380000,0.296296,0.100000


## Group F — Transcript Quality

This group measures how much of the transcript is marked as background
rather than direct tutor-student dialogue.

### Feature

20. `background_turn_ratio`  
    Proportion of all transcript turns assigned to the `BACKGROUND` role.

### Formula

```text
background_turn_ratio =
background_turns / total_turns

In [17]:
# =========================================================
# SECTION 6 — TRANSCRIPT QUALITY
# =========================================================
GROUP_F_FILE = OUTPUT_DIR / "group_f_transcript_quality.parquet"
FINAL_FEATURE_FILE = OUTPUT_DIR / "train_transcripts_baseline_20_features.parquet"

section_f_rows = []
start_time_f = perf_counter()

for file_number, file_path in enumerate(transcript_files, start=1):

    df = pd.read_csv(file_path, usecols=["session_id", "role"], dtype={"session_id": "string", "role": "string"})

    valid_session_ids = df["session_id"].dropna().astype(str).str.strip()
    session_id = valid_session_ids.iloc[0] if len(valid_session_ids) > 0 else file_path.stem

    role = df["role"].fillna("UNKNOWN").astype("string").str.strip().str.upper()

    total_turns = int(len(df))
    background_turns = int(role.eq("BACKGROUND").sum())
    background_turn_ratio = background_turns / total_turns if total_turns > 0 else 0.0

    section_f_rows.append({"session_id": session_id, "background_turn_ratio": background_turn_ratio})

    if file_number % 500 == 0 or file_number == len(transcript_files):
        elapsed = perf_counter() - start_time_f
        speed = file_number / elapsed if elapsed > 0 else 0
        remaining = len(transcript_files) - file_number
        eta_minutes = remaining / speed / 60 if speed > 0 else 0

        print(f"Processed {file_number:,}/{len(transcript_files):,} | ETA: {eta_minutes:.1f} min")


# =========================================================
# CREATE AND VALIDATE GROUP F TABLE
# =========================================================
group_f = pd.DataFrame(section_f_rows)

group_f["background_turn_ratio"] = group_f["background_turn_ratio"].replace([np.inf, -np.inf], 0.0).fillna(0.0).astype("float32")

if not group_f["session_id"].is_unique:
    duplicates = group_f.loc[group_f["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate session IDs found: {duplicates[:10]}")

if not group_f["background_turn_ratio"].between(0, 1).all():
    raise ValueError("Background turn ratio contains values outside the 0–1 range.")


# =========================================================
# SAVE GROUP F AND COMBINE ALL FEATURES
# =========================================================
group_f.to_parquet(GROUP_F_FILE, index=False, engine="pyarrow", compression="snappy")

session_features = session_features.merge(group_f, on="session_id", how="left", validate="one_to_one")


# =========================================================
# FINAL 20-FEATURE VALIDATION
# =========================================================
final_feature_columns = [
    "total_turns", "total_words", "session_duration_minutes", "turns_per_minute",
    "student_turns", "tutor_turns", "student_turn_ratio", "student_word_ratio",
    "avg_student_words_per_turn", "avg_tutor_words_per_turn",
    "student_short_turn_ratio", "student_long_turn_ratio",
    "student_numeric_turn_ratio", "student_question_ratio",
    "tutor_question_ratio", "student_response_after_tutor_question_ratio",
    "speaker_switch_rate", "longest_tutor_streak_ratio",
    "longest_student_streak_ratio", "background_turn_ratio",
]

missing_features = [column for column in final_feature_columns if column not in session_features.columns]

if missing_features:
    raise ValueError(f"Missing final features: {missing_features}")

session_features[final_feature_columns] = session_features[final_feature_columns].replace([np.inf, -np.inf], 0.0).fillna(0.0)

if not session_features["session_id"].is_unique:
    raise ValueError("Duplicate session IDs remain in the final feature table.")


# =========================================================
# SAVE FINAL 20-FEATURE DATASET
# =========================================================
session_features.to_parquet(FINAL_FEATURE_FILE, index=False, engine="pyarrow", compression="snappy")

elapsed_minutes_f = (perf_counter() - start_time_f) / 60

print("\nGroup F and final feature dataset completed successfully.")
print("Group F shape      :", group_f.shape)
print("Final table shape  :", session_features.shape)
print("Time               :", f"{elapsed_minutes_f:.2f} minutes")
print("Group F file       :", GROUP_F_FILE)
print("Final feature file :", FINAL_FEATURE_FILE)
print("Final file size    :", f"{FINAL_FEATURE_FILE.stat().st_size / 1024**2:.2f} MB")

display(session_features.head())

Processed 500/22,821 | ETA: 0.6 min
Processed 1,000/22,821 | ETA: 0.6 min
Processed 1,500/22,821 | ETA: 0.6 min
Processed 2,000/22,821 | ETA: 0.5 min
Processed 2,500/22,821 | ETA: 0.5 min
Processed 3,000/22,821 | ETA: 0.5 min
Processed 3,500/22,821 | ETA: 0.5 min
Processed 4,000/22,821 | ETA: 0.5 min
Processed 4,500/22,821 | ETA: 0.5 min
Processed 5,000/22,821 | ETA: 0.5 min
Processed 5,500/22,821 | ETA: 0.4 min
Processed 6,000/22,821 | ETA: 0.4 min
Processed 6,500/22,821 | ETA: 0.4 min
Processed 7,000/22,821 | ETA: 0.4 min
Processed 7,500/22,821 | ETA: 0.4 min
Processed 8,000/22,821 | ETA: 0.4 min
Processed 8,500/22,821 | ETA: 0.4 min
Processed 9,000/22,821 | ETA: 0.4 min
Processed 9,500/22,821 | ETA: 0.3 min
Processed 10,000/22,821 | ETA: 0.3 min
Processed 10,500/22,821 | ETA: 0.3 min
Processed 11,000/22,821 | ETA: 0.3 min
Processed 11,500/22,821 | ETA: 0.3 min
Processed 12,000/22,821 | ETA: 0.3 min
Processed 12,500/22,821 | ETA: 0.3 min
Processed 13,000/22,821 | ETA: 0.3 min
Process

,session_id,source_file,total_turns,total_words,session_duration_minutes,turns_per_minute,student_turns,tutor_turns,student_turn_ratio,student_word_ratio,...,student_short_turn_ratio,student_long_turn_ratio,student_numeric_turn_ratio,student_question_ratio,tutor_question_ratio,student_response_after_tutor_question_ratio,speaker_switch_rate,longest_tutor_streak_ratio,longest_student_streak_ratio,background_turn_ratio
0,aaaedit,aaaedit.csv,254,3156,43.816666,5.796881,114,136,0.448819,0.392269,...,0.429825,0.333333,0.236842,0.333333,0.507353,0.753623,0.779116,0.044118,0.035088,0.015748
1,aaaptjd,aaaptjd.csv,360,6376,45.500000,7.912088,178,165,0.494444,0.269134,...,0.471910,0.252809,0.516854,0.202247,0.721212,0.865546,0.862573,0.024242,0.022472,0.047222
2,aabkeov,aabkeov.csv,281,3045,36.799999,7.635870,136,137,0.483986,0.359606,...,0.529412,0.213235,0.205882,0.257353,0.583942,0.875000,0.849265,0.021898,0.029412,0.028470
3,aacggvb,aacggvb.csv,235,4180,46.266666,5.079251,112,118,0.476596,0.282775,...,0.553571,0.321429,0.223214,0.160714,0.627119,0.851351,0.816594,0.025424,0.026786,0.021277
4,aadexbc,aadexbc.csv,104,2676,44.033333,2.361847,20,81,0.192308,0.057175,...,0.600000,0.200000,0.150000,0.250000,0.777778,0.253968,0.380000,0.296296,0.100000,0.028846


## Load Training Features and Labels

This step loads the official training feature and label files.

- `train_features` contains the response ID, session ID, and learning objective.
- `train_labels` contains the response ID and correctness label.
- `response_id` is used to connect the two tables.
- Repeated `session_id` values are allowed because one tutoring session may have multiple assessment responses.

In [22]:
from pathlib import Path
import pandas as pd


# =========================================================
# FIND DATASET ROOT
# =========================================================
CURRENT_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = [CURRENT_DIR, CURRENT_DIR / "Trace-The-Race-Dataset", CURRENT_DIR.parent / "Trace-The-Race-Dataset"]

ROOT = next((path for path in ROOT_CANDIDATES if (path / "train_features_TMQTWsB.csv").exists()), None)

if ROOT is None:
    raise FileNotFoundError("Could not find the Trace-The-Race-Dataset folder.")

TRAIN_FEATURES_FILE = ROOT / "train_features_TMQTWsB.csv"
TRAIN_LABELS_FILE = ROOT / "train_labels_44ujmj2.csv"


# =========================================================
# LOAD FILES
# =========================================================
train_features = pd.read_csv(TRAIN_FEATURES_FILE)
train_labels = pd.read_csv(TRAIN_LABELS_FILE)


# =========================================================
# CLEAN COLUMN NAMES
# =========================================================
train_features.columns = train_features.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip().str.lower()
train_labels.columns = train_labels.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip().str.lower()

print("Train feature columns:", train_features.columns.tolist())
print("Train label columns  :", train_labels.columns.tolist())


# =========================================================
# FIND AND STANDARDIZE LABEL COLUMN
# =========================================================
possible_label_columns = ["correct", "label", "target", "is_correct", "answer_correct"]

label_column = next((column for column in possible_label_columns if column in train_labels.columns), None)

if label_column is None:
    raise ValueError(f"Could not identify the label column. Available columns: {train_labels.columns.tolist()}")

if label_column != "correct":
    train_labels = train_labels.rename(columns={label_column: "correct"})

train_features["response_id"] = train_features["response_id"].astype("string").str.strip()
train_features["session_id"] = train_features["session_id"].astype("string").str.strip()
train_features["learning_objective"] = train_features["learning_objective"].astype("string").str.strip()

train_labels["response_id"] = train_labels["response_id"].astype("string").str.strip()
train_labels["correct"] = pd.to_numeric(train_labels["correct"], errors="raise").astype("int8")


# =========================================================
# VALIDATION
# =========================================================
if train_features["response_id"].duplicated().any():
    raise ValueError("Duplicate response_id found in train_features.")

if train_labels["response_id"].duplicated().any():
    raise ValueError("Duplicate response_id found in train_labels.")

feature_ids = set(train_features["response_id"].dropna())
label_ids = set(train_labels["response_id"].dropna())


# =========================================================
# SUMMARY
# =========================================================
print("\nTRAIN FEATURES")
print("Shape                 :", train_features.shape)
print("Unique response_id    :", train_features["response_id"].nunique())
print("Unique session_id     :", train_features["session_id"].nunique())
print("Repeated session rows :", train_features["session_id"].duplicated().sum())

print("\nTRAIN LABELS")
print("Shape                 :", train_labels.shape)
print("Unique response_id    :", train_labels["response_id"].nunique())
print("Label column          :", "correct")
print("Label distribution:")
print(train_labels["correct"].value_counts(dropna=False).sort_index())

print("\nID ALIGNMENT")
print("Features without labels :", len(feature_ids - label_ids))
print("Labels without features :", len(label_ids - feature_ids))

display(train_features.head())
display(train_labels.head())

Train feature columns: ['response_id', 'session_id', 'learning_objective_id', 'learning_objective']
Train label columns  : ['response_id', 'is_correct']

TRAIN FEATURES
Shape                 : (35072, 4)
Unique response_id    : 35072
Unique session_id     : 22821
Repeated session rows : 12251

TRAIN LABELS
Shape                 : (35072, 2)
Unique response_id    : 35072
Label column          : correct
Label distribution:
correct
0    10435
1    24637
Name: count, dtype: int64

ID ALIGNMENT
Features without labels : 0
Labels without features : 0


,response_id,session_id,learning_objective_id,learning_objective
0,aaaavsh,bcaufvc,dqibnvd,Knowing the value of each digit in numbers wit...
1,aaabhzi,eyutanf,eukmzxl,Adding and subtracting tens to a 2-digit number.
2,aaahpnz,juptkxd,fjbqcsv,Comparing and ordering fractions by finding a ...
3,aaajpom,ntwkcfj,acvbcev,Comparing fractions using reasoning.
4,aaamwux,jqriibm,krfuudx,Counting in multiples.


,response_id,correct
0,aaaavsh,1
1,aaabhzi,1
2,aaahpnz,0
3,aaajpom,0
4,aaamwux,0


## Final Session-Level Transcript Dataset

This step creates the complete session-level transcript dataset.

Each row represents one tutoring session and contains:

- `session_id`
- Source transcript filename
- Full tagged transcript
- Student-only text
- Tutor-only text
- Twenty numerical transcript features
- Seven threshold-based categorical features

### Threshold Features

21. `session_turn_volume_band`
22. `session_duration_band`
23. `student_turn_share_band`
24. `student_response_length_band`
25. `student_short_turn_band`
26. `tutor_questioning_band`
27. `dialogue_switch_band`

The categories are:

```text
LOW
MEDIUM
HIGH

In [23]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd


# =========================================================
# 1. FIND DATASET AND OUTPUT PATHS
# =========================================================
CURRENT_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = [CURRENT_DIR, CURRENT_DIR / "Trace-The-Race-Dataset", CURRENT_DIR.parent / "Trace-The-Race-Dataset"]
ROOT = next((path for path in ROOT_CANDIDATES if (path / "train_transcripts").is_dir()), None)

if ROOT is None:
    raise FileNotFoundError("Trace-The-Race-Dataset/train_transcripts folder was not found.")

TRANSCRIPT_DIR = ROOT / "train_transcripts"
OUTPUT_DIR = ROOT / "outputs" / "02_feature_groups"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILE = OUTPUT_DIR / "train_transcripts_baseline_20_features.parquet"
FINAL_TRANSCRIPT_FILE = OUTPUT_DIR / "train_transcripts_full_27_features.parquet"
THRESHOLD_FILE = OUTPUT_DIR / "threshold_definitions.csv"

transcript_files = sorted(TRANSCRIPT_DIR.glob("*.csv"))

if not transcript_files:
    raise FileNotFoundError(f"No transcript CSV files found inside: {TRANSCRIPT_DIR}")


# =========================================================
# 2. LOAD THE PREVIOUSLY CREATED 20 FEATURES
# =========================================================
if "session_features" not in globals():
    if not FEATURE_FILE.exists():
        raise FileNotFoundError(f"20-feature file was not found: {FEATURE_FILE}")

    session_features = pd.read_parquet(FEATURE_FILE)

session_features["session_id"] = session_features["session_id"].astype("string").str.strip()

if session_features["session_id"].duplicated().any():
    raise ValueError("Duplicate session_id found in the 20-feature dataset.")

print("Transcript files :", f"{len(transcript_files):,}")
print("Feature rows     :", f"{len(session_features):,}")


# =========================================================
# 3. CREATE ONE FULL TEXT ROW FOR EACH SESSION
# =========================================================
transcript_rows = []
start_time = perf_counter()

for file_number, file_path in enumerate(transcript_files, start=1):

    df = pd.read_csv(file_path, usecols=["session_id", "utterance_id", "role", "content"], dtype={"session_id": "string", "role": "string", "content": "string"})

    valid_session_ids = df["session_id"].dropna().astype("string").str.strip()
    session_id = valid_session_ids.iloc[0] if len(valid_session_ids) > 0 else file_path.stem

    df["_order"] = pd.to_numeric(df["utterance_id"], errors="coerce")
    df = df.sort_values("_order", kind="stable", na_position="last").reset_index(drop=True)

    role = df["role"].fillna("UNKNOWN").astype("string").str.strip().str.upper()
    content = df["content"].fillna("").astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

    valid_content = content.ne("")
    tagged_turns = "[" + role + "] " + content

    transcript_text = "\n".join(tagged_turns[valid_content].tolist())
    student_text = "\n".join(content[role.eq("STUDENT") & valid_content].tolist())
    tutor_text = "\n".join(content[role.eq("TUTOR") & valid_content].tolist())

    transcript_rows.append({"session_id": session_id, "source_file": file_path.name, "transcript_text": transcript_text, "student_text": student_text, "tutor_text": tutor_text})

    if file_number % 500 == 0 or file_number == len(transcript_files):
        elapsed = perf_counter() - start_time
        speed = file_number / elapsed if elapsed > 0 else 0
        remaining = len(transcript_files) - file_number
        eta_minutes = remaining / speed / 60 if speed > 0 else 0

        print(f"Processed {file_number:,}/{len(transcript_files):,} | ETA: {eta_minutes:.1f} min")


transcript_table = pd.DataFrame(transcript_rows)
transcript_table["session_id"] = transcript_table["session_id"].astype("string").str.strip()

if transcript_table["session_id"].duplicated().any():
    duplicates = transcript_table.loc[transcript_table["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate transcript session IDs found: {duplicates[:10]}")


# =========================================================
# 4. MERGE TEXT WITH THE 20 NUMERICAL FEATURES
# =========================================================
feature_table = session_features.drop(columns=["source_file"], errors="ignore")

full_transcript_features = transcript_table.merge(feature_table, on="session_id", how="left", validate="one_to_one", indicator="_feature_merge")

missing_feature_rows = int(full_transcript_features["_feature_merge"].eq("left_only").sum())

if missing_feature_rows > 0:
    raise ValueError(f"{missing_feature_rows:,} transcript sessions do not have numerical features.")

full_transcript_features = full_transcript_features.drop(columns="_feature_merge")


# =========================================================
# 5. FUNCTION FOR LOW / MEDIUM / HIGH BANDS
# =========================================================
def create_threshold_band(series):

    values = pd.to_numeric(series, errors="coerce").fillna(0.0)
    q33 = float(values.quantile(1 / 3))
    q67 = float(values.quantile(2 / 3))

    if q33 < q67:
        bands = np.select([values <= q33, values <= q67], ["LOW", "MEDIUM"], default="HIGH")
        return pd.Series(bands, index=series.index, dtype="string"), "tertile", q33, q67

    positive_values = values[values > 0]

    if values.eq(0).any() and positive_values.nunique() >= 2:
        positive_median = float(positive_values.median())
        bands = np.where(values.eq(0), "LOW", np.where(values <= positive_median, "MEDIUM", "HIGH"))
        return pd.Series(bands, index=series.index, dtype="string"), "zero_positive_median", 0.0, positive_median

    if values.nunique() == 1:
        bands = pd.Series("MEDIUM", index=series.index, dtype="string")
        return bands, "single_value", q33, q67

    median_value = float(values.median())
    bands = np.where(values < median_value, "LOW", np.where(values > median_value, "HIGH", "MEDIUM"))
    return pd.Series(bands, index=series.index, dtype="string"), "median_fallback", median_value, median_value


# =========================================================
# 6. CREATE THE 7 THRESHOLD FEATURES
# =========================================================
band_definitions = {
    "session_turn_volume_band": "total_turns",
    "session_duration_band": "session_duration_minutes",
    "student_turn_share_band": "student_turn_ratio",
    "student_response_length_band": "avg_student_words_per_turn",
    "student_short_turn_band": "student_short_turn_ratio",
    "tutor_questioning_band": "tutor_question_ratio",
    "dialogue_switch_band": "speaker_switch_rate",
}

threshold_rows = []

for band_column, source_column in band_definitions.items():

    if source_column not in full_transcript_features.columns:
        raise ValueError(f"Required source feature is missing: {source_column}")

    bands, method, low_upper, medium_upper = create_threshold_band(full_transcript_features[source_column])
    full_transcript_features[band_column] = bands

    threshold_rows.append({
        "band_feature": band_column,
        "source_feature": source_column,
        "method": method,
        "low_upper_threshold": low_upper,
        "medium_upper_threshold": medium_upper,
    })


threshold_definitions = pd.DataFrame(threshold_rows)


# =========================================================
# 7. FINAL COLUMN ORDER
# =========================================================
numerical_features = [
    "total_turns", "total_words", "session_duration_minutes", "turns_per_minute",
    "student_turns", "tutor_turns", "student_turn_ratio", "student_word_ratio",
    "avg_student_words_per_turn", "avg_tutor_words_per_turn",
    "student_short_turn_ratio", "student_long_turn_ratio",
    "student_numeric_turn_ratio", "student_question_ratio",
    "tutor_question_ratio", "student_response_after_tutor_question_ratio",
    "speaker_switch_rate", "longest_tutor_streak_ratio",
    "longest_student_streak_ratio", "background_turn_ratio",
]

threshold_features = list(band_definitions.keys())
identifier_and_text_columns = ["session_id", "source_file", "transcript_text", "student_text", "tutor_text"]

final_columns = identifier_and_text_columns + numerical_features + threshold_features
missing_columns = [column for column in final_columns if column not in full_transcript_features.columns]

if missing_columns:
    raise ValueError(f"Final dataset is missing columns: {missing_columns}")

full_transcript_features = full_transcript_features[final_columns].copy()


# =========================================================
# 8. FINAL VALIDATION
# =========================================================
if not full_transcript_features["session_id"].is_unique:
    raise ValueError("Final transcript dataset contains duplicate session IDs.")

if full_transcript_features[numerical_features].isna().any().any():
    raise ValueError("Missing values found in the numerical features.")

valid_band_values = {"LOW", "MEDIUM", "HIGH"}

for column in threshold_features:
    invalid_values = set(full_transcript_features[column].dropna().unique()) - valid_band_values

    if invalid_values:
        raise ValueError(f"Invalid categories found in {column}: {invalid_values}")


# =========================================================
# 9. SAVE FULL SESSION-LEVEL PARQUET AND THRESHOLDS
# =========================================================
full_transcript_features.to_parquet(FINAL_TRANSCRIPT_FILE, index=False, engine="pyarrow", compression="snappy")
threshold_definitions.to_csv(THRESHOLD_FILE, index=False)

elapsed_minutes = (perf_counter() - start_time) / 60

print("\nFull transcript feature dataset completed successfully.")
print("Final rows       :", f"{len(full_transcript_features):,}")
print("Final columns    :", full_transcript_features.shape[1])
print("Numerical fields :", len(numerical_features))
print("Threshold fields :", len(threshold_features))
print("Processing time  :", f"{elapsed_minutes:.2f} minutes")
print("Parquet file     :", FINAL_TRANSCRIPT_FILE)
print("Threshold file   :", THRESHOLD_FILE)
print("Parquet size     :", f"{FINAL_TRANSCRIPT_FILE.stat().st_size / 1024**2:.2f} MB")

print("\nThreshold definitions:")
display(threshold_definitions)

print("\nFinal dataset preview:")
display(full_transcript_features.head())

Transcript files : 22,821
Feature rows     : 22,821
Processed 500/22,821 | ETA: 1.3 min
Processed 1,000/22,821 | ETA: 1.3 min
Processed 1,500/22,821 | ETA: 1.3 min
Processed 2,000/22,821 | ETA: 1.2 min
Processed 2,500/22,821 | ETA: 1.2 min
Processed 3,000/22,821 | ETA: 1.2 min
Processed 3,500/22,821 | ETA: 1.1 min
Processed 4,000/22,821 | ETA: 1.1 min
Processed 4,500/22,821 | ETA: 1.1 min
Processed 5,000/22,821 | ETA: 1.1 min
Processed 5,500/22,821 | ETA: 1.0 min
Processed 6,000/22,821 | ETA: 1.0 min
Processed 6,500/22,821 | ETA: 1.0 min
Processed 7,000/22,821 | ETA: 0.9 min
Processed 7,500/22,821 | ETA: 0.9 min
Processed 8,000/22,821 | ETA: 0.9 min
Processed 8,500/22,821 | ETA: 0.9 min
Processed 9,000/22,821 | ETA: 0.8 min
Processed 9,500/22,821 | ETA: 0.8 min
Processed 10,000/22,821 | ETA: 0.8 min
Processed 10,500/22,821 | ETA: 0.7 min
Processed 11,000/22,821 | ETA: 0.7 min
Processed 11,500/22,821 | ETA: 0.7 min
Processed 12,000/22,821 | ETA: 0.6 min
Processed 12,500/22,821 | ETA: 0.

,band_feature,source_feature,method,low_upper_threshold,medium_upper_threshold
0,session_turn_volume_band,total_turns,tertile,238.000000,298.000000
1,session_duration_band,session_duration_minutes,tertile,40.900002,44.799999
2,student_turn_share_band,student_turn_ratio,tertile,0.423898,0.463478
3,student_response_length_band,avg_student_words_per_turn,tertile,6.831222,9.095897
4,student_short_turn_band,student_short_turn_ratio,tertile,0.452381,0.558376
5,tutor_questioning_band,tutor_question_ratio,tertile,0.575758,0.688089
6,dialogue_switch_band,speaker_switch_rate,tertile,0.779006,0.831373



Final dataset preview:


,session_id,source_file,transcript_text,student_text,tutor_text,total_turns,total_words,session_duration_minutes,turns_per_minute,student_turns,...,longest_tutor_streak_ratio,longest_student_streak_ratio,background_turn_ratio,session_turn_volume_band,session_duration_band,student_turn_share_band,student_response_length_band,student_short_turn_band,tutor_questioning_band,dialogue_switch_band
0,aaaedit,aaaedit.csv,[TUTOR] Hello?\n[BACKGROUND] [unclear]\n[BACKG...,"Good.\nGood.\nDuring the weekend, on the 15th ...","Hello?\nHi, Lachlan. How are you doing today?\...",254,3156,43.816666,5.796881,114,...,0.044118,0.035088,0.015748,MEDIUM,MEDIUM,MEDIUM,HIGH,LOW,LOW,MEDIUM
1,aaaptjd,aaaptjd.csv,[BACKGROUND] [unclear]\n[TUTOR] Hello?\n[STUDE...,Hello?\nI'm good.\nAdding and subtracting amou...,"Hello?\nHi, how are you doing today?\nOkay, I ...",360,6376,45.500000,7.912088,178,...,0.024242,0.022472,0.047222,HIGH,HIGH,HIGH,HIGH,MEDIUM,HIGH,HIGH
2,aabkeov,aabkeov.csv,[STUDENT] Hello.\n[BACKGROUND] [unclear]\n[TUT...,"Hello.\nHi.\nThank you.\nHello?\nYeah, how are...","Hello.\nWelcome back.\nHi, Kaelan. [unclear]\n...",281,3045,36.799999,7.635870,136,...,0.021898,0.029412,0.028470,MEDIUM,LOW,HIGH,MEDIUM,MEDIUM,MEDIUM,HIGH
3,aacggvb,aacggvb.csv,"[BACKGROUND] [unclear]\n[TUTOR] Hello? Tobias,...","Uh, yeah.\nYes.\nGood.\nHuh?\nYeah, that's why...","Hello? Tobias, can you hear me?\nOkay. So we h...",235,4180,46.266666,5.079251,112,...,0.025424,0.026786,0.021277,LOW,HIGH,HIGH,HIGH,MEDIUM,MEDIUM,MEDIUM
4,aadexbc,aadexbc.csv,"[BACKGROUND] [unclear]\n[TUTOR] Hi. Bryony, ca...","Hello.\nI'm good, how are you?\nGood.\nYeah.\n...","Hi. Bryony, can you hear me?\nHi Bryony, how a...",104,2676,44.033333,2.361847,20,...,0.296296,0.100000,0.028846,LOW,MEDIUM,LOW,MEDIUM,HIGH,HIGH,LOW


In [24]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd


# =========================================================
# 1. FIND DATASET AND OUTPUT PATHS
# =========================================================
CURRENT_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = [CURRENT_DIR, CURRENT_DIR / "Trace-The-Race-Dataset", CURRENT_DIR.parent / "Trace-The-Race-Dataset"]
ROOT = next((path for path in ROOT_CANDIDATES if (path / "train_transcripts").is_dir()), None)

if ROOT is None:
    raise FileNotFoundError("Trace-The-Race-Dataset/train_transcripts folder was not found.")

TRANSCRIPT_DIR = ROOT / "train_transcripts"
OUTPUT_DIR = ROOT / "outputs" / "02_feature_groups"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILE = OUTPUT_DIR / "train_transcripts_baseline_20_features.parquet"
FINAL_TRANSCRIPT_FILE = OUTPUT_DIR / "train_transcripts_full_27_features.parquet"
THRESHOLD_FILE = OUTPUT_DIR / "threshold_definitions.csv"

transcript_files = sorted(TRANSCRIPT_DIR.glob("*.csv"))

if not transcript_files:
    raise FileNotFoundError(f"No transcript CSV files found inside: {TRANSCRIPT_DIR}")


# =========================================================
# 2. LOAD THE PREVIOUSLY CREATED 20 FEATURES
# =========================================================
if "session_features" not in globals():
    if not FEATURE_FILE.exists():
        raise FileNotFoundError(f"20-feature file was not found: {FEATURE_FILE}")

    session_features = pd.read_parquet(FEATURE_FILE)

session_features["session_id"] = session_features["session_id"].astype("string").str.strip()

if session_features["session_id"].duplicated().any():
    raise ValueError("Duplicate session_id found in the 20-feature dataset.")

print("Transcript files :", f"{len(transcript_files):,}")
print("Feature rows     :", f"{len(session_features):,}")


# =========================================================
# 3. CREATE ONE FULL TEXT ROW FOR EACH SESSION
# =========================================================
transcript_rows = []
start_time = perf_counter()

for file_number, file_path in enumerate(transcript_files, start=1):

    df = pd.read_csv(file_path, usecols=["session_id", "utterance_id", "role", "content"], dtype={"session_id": "string", "role": "string", "content": "string"})

    valid_session_ids = df["session_id"].dropna().astype("string").str.strip()
    session_id = valid_session_ids.iloc[0] if len(valid_session_ids) > 0 else file_path.stem

    df["_order"] = pd.to_numeric(df["utterance_id"], errors="coerce")
    df = df.sort_values("_order", kind="stable", na_position="last").reset_index(drop=True)

    role = df["role"].fillna("UNKNOWN").astype("string").str.strip().str.upper()
    content = df["content"].fillna("").astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

    valid_content = content.ne("")
    tagged_turns = "[" + role + "] " + content

    transcript_text = "\n".join(tagged_turns[valid_content].tolist())
    student_text = "\n".join(content[role.eq("STUDENT") & valid_content].tolist())
    tutor_text = "\n".join(content[role.eq("TUTOR") & valid_content].tolist())

    transcript_rows.append({"session_id": session_id, "source_file": file_path.name, "transcript_text": transcript_text, "student_text": student_text, "tutor_text": tutor_text})

    if file_number % 500 == 0 or file_number == len(transcript_files):
        elapsed = perf_counter() - start_time
        speed = file_number / elapsed if elapsed > 0 else 0
        remaining = len(transcript_files) - file_number
        eta_minutes = remaining / speed / 60 if speed > 0 else 0

        print(f"Processed {file_number:,}/{len(transcript_files):,} | ETA: {eta_minutes:.1f} min")


transcript_table = pd.DataFrame(transcript_rows)
transcript_table["session_id"] = transcript_table["session_id"].astype("string").str.strip()

if transcript_table["session_id"].duplicated().any():
    duplicates = transcript_table.loc[transcript_table["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate transcript session IDs found: {duplicates[:10]}")


# =========================================================
# 4. MERGE TEXT WITH THE 20 NUMERICAL FEATURES
# =========================================================
feature_table = session_features.drop(columns=["source_file"], errors="ignore")

full_transcript_features = transcript_table.merge(feature_table, on="session_id", how="left", validate="one_to_one", indicator="_feature_merge")

missing_feature_rows = int(full_transcript_features["_feature_merge"].eq("left_only").sum())

if missing_feature_rows > 0:
    raise ValueError(f"{missing_feature_rows:,} transcript sessions do not have numerical features.")

full_transcript_features = full_transcript_features.drop(columns="_feature_merge")


# =========================================================
# 5. FUNCTION FOR LOW / MEDIUM / HIGH BANDS
# =========================================================
def create_threshold_band(series):

    values = pd.to_numeric(series, errors="coerce").fillna(0.0)
    q33 = float(values.quantile(1 / 3))
    q67 = float(values.quantile(2 / 3))

    if q33 < q67:
        bands = np.select([values <= q33, values <= q67], ["LOW", "MEDIUM"], default="HIGH")
        return pd.Series(bands, index=series.index, dtype="string"), "tertile", q33, q67

    positive_values = values[values > 0]

    if values.eq(0).any() and positive_values.nunique() >= 2:
        positive_median = float(positive_values.median())
        bands = np.where(values.eq(0), "LOW", np.where(values <= positive_median, "MEDIUM", "HIGH"))
        return pd.Series(bands, index=series.index, dtype="string"), "zero_positive_median", 0.0, positive_median

    if values.nunique() == 1:
        bands = pd.Series("MEDIUM", index=series.index, dtype="string")
        return bands, "single_value", q33, q67

    median_value = float(values.median())
    bands = np.where(values < median_value, "LOW", np.where(values > median_value, "HIGH", "MEDIUM"))
    return pd.Series(bands, index=series.index, dtype="string"), "median_fallback", median_value, median_value


# =========================================================
# 6. CREATE THE 7 THRESHOLD FEATURES
# =========================================================
band_definitions = {
    "session_turn_volume_band": "total_turns",
    "session_duration_band": "session_duration_minutes",
    "student_turn_share_band": "student_turn_ratio",
    "student_response_length_band": "avg_student_words_per_turn",
    "student_short_turn_band": "student_short_turn_ratio",
    "tutor_questioning_band": "tutor_question_ratio",
    "dialogue_switch_band": "speaker_switch_rate",
}

threshold_rows = []

for band_column, source_column in band_definitions.items():

    if source_column not in full_transcript_features.columns:
        raise ValueError(f"Required source feature is missing: {source_column}")

    bands, method, low_upper, medium_upper = create_threshold_band(full_transcript_features[source_column])
    full_transcript_features[band_column] = bands

    threshold_rows.append({
        "band_feature": band_column,
        "source_feature": source_column,
        "method": method,
        "low_upper_threshold": low_upper,
        "medium_upper_threshold": medium_upper,
    })


threshold_definitions = pd.DataFrame(threshold_rows)


# =========================================================
# 7. FINAL COLUMN ORDER
# =========================================================
numerical_features = [
    "total_turns", "total_words", "session_duration_minutes", "turns_per_minute",
    "student_turns", "tutor_turns", "student_turn_ratio", "student_word_ratio",
    "avg_student_words_per_turn", "avg_tutor_words_per_turn",
    "student_short_turn_ratio", "student_long_turn_ratio",
    "student_numeric_turn_ratio", "student_question_ratio",
    "tutor_question_ratio", "student_response_after_tutor_question_ratio",
    "speaker_switch_rate", "longest_tutor_streak_ratio",
    "longest_student_streak_ratio", "background_turn_ratio",
]

threshold_features = list(band_definitions.keys())
identifier_and_text_columns = ["session_id", "source_file", "transcript_text", "student_text", "tutor_text"]

final_columns = identifier_and_text_columns + numerical_features + threshold_features
missing_columns = [column for column in final_columns if column not in full_transcript_features.columns]

if missing_columns:
    raise ValueError(f"Final dataset is missing columns: {missing_columns}")

full_transcript_features = full_transcript_features[final_columns].copy()


# =========================================================
# 8. FINAL VALIDATION
# =========================================================
if not full_transcript_features["session_id"].is_unique:
    raise ValueError("Final transcript dataset contains duplicate session IDs.")

if full_transcript_features[numerical_features].isna().any().any():
    raise ValueError("Missing values found in the numerical features.")

valid_band_values = {"LOW", "MEDIUM", "HIGH"}

for column in threshold_features:
    invalid_values = set(full_transcript_features[column].dropna().unique()) - valid_band_values

    if invalid_values:
        raise ValueError(f"Invalid categories found in {column}: {invalid_values}")


# =========================================================
# 9. SAVE FULL SESSION-LEVEL PARQUET AND THRESHOLDS
# =========================================================
full_transcript_features.to_parquet(FINAL_TRANSCRIPT_FILE, index=False, engine="pyarrow", compression="snappy")
threshold_definitions.to_csv(THRESHOLD_FILE, index=False)

elapsed_minutes = (perf_counter() - start_time) / 60

print("\nFull transcript feature dataset completed successfully.")
print("Final rows       :", f"{len(full_transcript_features):,}")
print("Final columns    :", full_transcript_features.shape[1])
print("Numerical fields :", len(numerical_features))
print("Threshold fields :", len(threshold_features))
print("Processing time  :", f"{elapsed_minutes:.2f} minutes")
print("Parquet file     :", FINAL_TRANSCRIPT_FILE)
print("Threshold file   :", THRESHOLD_FILE)
print("Parquet size     :", f"{FINAL_TRANSCRIPT_FILE.stat().st_size / 1024**2:.2f} MB")

print("\nThreshold definitions:")
display(threshold_definitions)

print("\nFinal dataset preview:")
display(full_transcript_features.head())

Transcript files : 22,821
Feature rows     : 22,821
Processed 500/22,821 | ETA: 1.4 min
Processed 1,000/22,821 | ETA: 1.3 min
Processed 1,500/22,821 | ETA: 1.3 min
Processed 2,000/22,821 | ETA: 1.3 min
Processed 2,500/22,821 | ETA: 1.2 min
Processed 3,000/22,821 | ETA: 1.2 min
Processed 3,500/22,821 | ETA: 1.2 min
Processed 4,000/22,821 | ETA: 1.1 min
Processed 4,500/22,821 | ETA: 1.1 min
Processed 5,000/22,821 | ETA: 1.1 min
Processed 5,500/22,821 | ETA: 1.1 min
Processed 6,000/22,821 | ETA: 1.0 min
Processed 6,500/22,821 | ETA: 1.0 min
Processed 7,000/22,821 | ETA: 1.0 min
Processed 7,500/22,821 | ETA: 0.9 min
Processed 8,000/22,821 | ETA: 0.9 min
Processed 8,500/22,821 | ETA: 0.9 min
Processed 9,000/22,821 | ETA: 0.8 min
Processed 9,500/22,821 | ETA: 0.8 min
Processed 10,000/22,821 | ETA: 0.8 min
Processed 10,500/22,821 | ETA: 0.7 min
Processed 11,000/22,821 | ETA: 0.7 min
Processed 11,500/22,821 | ETA: 0.7 min
Processed 12,000/22,821 | ETA: 0.7 min
Processed 12,500/22,821 | ETA: 0.

,band_feature,source_feature,method,low_upper_threshold,medium_upper_threshold
0,session_turn_volume_band,total_turns,tertile,238.000000,298.000000
1,session_duration_band,session_duration_minutes,tertile,40.900002,44.799999
2,student_turn_share_band,student_turn_ratio,tertile,0.423898,0.463478
3,student_response_length_band,avg_student_words_per_turn,tertile,6.831222,9.095897
4,student_short_turn_band,student_short_turn_ratio,tertile,0.452381,0.558376
5,tutor_questioning_band,tutor_question_ratio,tertile,0.575758,0.688089
6,dialogue_switch_band,speaker_switch_rate,tertile,0.779006,0.831373



Final dataset preview:


,session_id,source_file,transcript_text,student_text,tutor_text,total_turns,total_words,session_duration_minutes,turns_per_minute,student_turns,...,longest_tutor_streak_ratio,longest_student_streak_ratio,background_turn_ratio,session_turn_volume_band,session_duration_band,student_turn_share_band,student_response_length_band,student_short_turn_band,tutor_questioning_band,dialogue_switch_band
0,aaaedit,aaaedit.csv,[TUTOR] Hello?\n[BACKGROUND] [unclear]\n[BACKG...,"Good.\nGood.\nDuring the weekend, on the 15th ...","Hello?\nHi, Lachlan. How are you doing today?\...",254,3156,43.816666,5.796881,114,...,0.044118,0.035088,0.015748,MEDIUM,MEDIUM,MEDIUM,HIGH,LOW,LOW,MEDIUM
1,aaaptjd,aaaptjd.csv,[BACKGROUND] [unclear]\n[TUTOR] Hello?\n[STUDE...,Hello?\nI'm good.\nAdding and subtracting amou...,"Hello?\nHi, how are you doing today?\nOkay, I ...",360,6376,45.500000,7.912088,178,...,0.024242,0.022472,0.047222,HIGH,HIGH,HIGH,HIGH,MEDIUM,HIGH,HIGH
2,aabkeov,aabkeov.csv,[STUDENT] Hello.\n[BACKGROUND] [unclear]\n[TUT...,"Hello.\nHi.\nThank you.\nHello?\nYeah, how are...","Hello.\nWelcome back.\nHi, Kaelan. [unclear]\n...",281,3045,36.799999,7.635870,136,...,0.021898,0.029412,0.028470,MEDIUM,LOW,HIGH,MEDIUM,MEDIUM,MEDIUM,HIGH
3,aacggvb,aacggvb.csv,"[BACKGROUND] [unclear]\n[TUTOR] Hello? Tobias,...","Uh, yeah.\nYes.\nGood.\nHuh?\nYeah, that's why...","Hello? Tobias, can you hear me?\nOkay. So we h...",235,4180,46.266666,5.079251,112,...,0.025424,0.026786,0.021277,LOW,HIGH,HIGH,HIGH,MEDIUM,MEDIUM,MEDIUM
4,aadexbc,aadexbc.csv,"[BACKGROUND] [unclear]\n[TUTOR] Hi. Bryony, ca...","Hello.\nI'm good, how are you?\nGood.\nYeah.\n...","Hi. Bryony, can you hear me?\nHi Bryony, how a...",104,2676,44.033333,2.361847,20,...,0.296296,0.100000,0.028846,LOW,MEDIUM,LOW,MEDIUM,HIGH,HIGH,LOW


## Create the Final Model-Ready Master Dataset

This step combines the official training data with the complete
session-level transcript feature table.

### Merge Process

1. `train_features` and `train_labels` are merged using `response_id`.
2. The result is merged with the transcript feature table using `session_id`.
3. `train_features` remains the base table so that no assessment response is lost.

### Merge Relationships

```text
train_features + train_labels
response_id → one-to-one

training table + transcript features
session_id → many-to-one

In [25]:
from pathlib import Path

import numpy as np
import pandas as pd


# =========================================================
# 1. FIND DATASET ROOT
# =========================================================
CURRENT_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = [CURRENT_DIR, CURRENT_DIR / "Trace-The-Race-Dataset", CURRENT_DIR.parent / "Trace-The-Race-Dataset"]
ROOT = next((path for path in ROOT_CANDIDATES if (path / "train_features_TMQTWsB.csv").exists()), None)

if ROOT is None:
    raise FileNotFoundError("Could not find the Trace-The-Race-Dataset folder.")


# =========================================================
# 2. DEFINE INPUT AND OUTPUT FILES
# =========================================================
TRAIN_FEATURES_FILE = ROOT / "train_features_TMQTWsB.csv"
TRAIN_LABELS_FILE = ROOT / "train_labels_44ujmj2.csv"
TRANSCRIPT_FEATURE_FILE = ROOT / "outputs" / "02_feature_groups" / "train_transcripts_full_27_features.parquet"

MASTER_OUTPUT_DIR = ROOT / "outputs" / "03_master_dataset"
MASTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_TRAIN_FILE = MASTER_OUTPUT_DIR / "master_train.parquet"

if not TRANSCRIPT_FEATURE_FILE.exists():
    raise FileNotFoundError(f"Transcript feature file was not found: {TRANSCRIPT_FEATURE_FILE}")


# =========================================================
# 3. LOAD ALL THREE TABLES
# =========================================================
train_features = pd.read_csv(TRAIN_FEATURES_FILE)
train_labels = pd.read_csv(TRAIN_LABELS_FILE)
transcript_features = pd.read_parquet(TRANSCRIPT_FEATURE_FILE)


# =========================================================
# 4. CLEAN COLUMN NAMES
# =========================================================
train_features.columns = train_features.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip().str.lower()
train_labels.columns = train_labels.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip().str.lower()
transcript_features.columns = transcript_features.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip().str.lower()


# =========================================================
# 5. IDENTIFY AND STANDARDIZE LABEL COLUMN
# =========================================================
possible_label_columns = ["correct", "label", "target", "is_correct", "answer_correct"]
label_column = next((column for column in possible_label_columns if column in train_labels.columns), None)

if label_column is None:
    non_id_columns = [column for column in train_labels.columns if column != "response_id"]

    if len(non_id_columns) == 1:
        label_column = non_id_columns[0]
    else:
        raise ValueError(f"Could not identify the label column. Available columns: {train_labels.columns.tolist()}")

if label_column != "correct":
    train_labels = train_labels.rename(columns={label_column: "correct"})


# =========================================================
# 6. CLEAN MERGE KEYS
# =========================================================
train_features["response_id"] = train_features["response_id"].astype("string").str.strip()
train_features["session_id"] = train_features["session_id"].astype("string").str.strip()

train_labels["response_id"] = train_labels["response_id"].astype("string").str.strip()
train_labels["correct"] = pd.to_numeric(train_labels["correct"], errors="raise").astype("int8")

transcript_features["session_id"] = transcript_features["session_id"].astype("string").str.strip()


# =========================================================
# 7. VALIDATE UNIQUE KEYS
# =========================================================
if train_features["response_id"].duplicated().any():
    duplicates = train_features.loc[train_features["response_id"].duplicated(), "response_id"].tolist()
    raise ValueError(f"Duplicate response_id found in train_features: {duplicates[:10]}")

if train_labels["response_id"].duplicated().any():
    duplicates = train_labels.loc[train_labels["response_id"].duplicated(), "response_id"].tolist()
    raise ValueError(f"Duplicate response_id found in train_labels: {duplicates[:10]}")

if transcript_features["session_id"].duplicated().any():
    duplicates = transcript_features.loc[transcript_features["session_id"].duplicated(), "session_id"].tolist()
    raise ValueError(f"Duplicate session_id found in transcript features: {duplicates[:10]}")

if not train_labels["correct"].isin([0, 1]).all():
    raise ValueError("The correct label contains values other than 0 and 1.")


# =========================================================
# 8. MERGE TRAIN FEATURES WITH LABELS
# =========================================================
master_train = train_features.merge(train_labels[["response_id", "correct"]], on="response_id", how="left", validate="one_to_one", indicator="_label_merge")

label_merge_summary = master_train["_label_merge"].value_counts()
missing_labels = int(master_train["_label_merge"].eq("left_only").sum())

print("Label merge status:")
print(label_merge_summary)

if missing_labels > 0:
    raise ValueError(f"{missing_labels:,} train feature rows do not have labels.")

master_train = master_train.drop(columns="_label_merge")


# =========================================================
# 9. MERGE THE FULL TRANSCRIPT FEATURE TABLE
# =========================================================
master_train = master_train.merge(transcript_features, on="session_id", how="left", validate="many_to_one", indicator="_transcript_merge")

transcript_merge_summary = master_train["_transcript_merge"].value_counts()
missing_transcript_rows = int(master_train["_transcript_merge"].eq("left_only").sum())

print("\nTranscript merge status:")
print(transcript_merge_summary)

master_train = master_train.drop(columns="_transcript_merge")


# =========================================================
# 10. HANDLE MISSING TRANSCRIPT DATA
# =========================================================
text_columns = [column for column in ["source_file", "transcript_text", "student_text", "tutor_text"] if column in master_train.columns]
band_columns = [column for column in master_train.columns if column.endswith("_band")]

protected_columns = {"response_id", "session_id", "learning_objective", "correct"}
numerical_feature_columns = [column for column in transcript_features.select_dtypes(include=[np.number]).columns if column not in protected_columns]

for column in text_columns:
    master_train[column] = master_train[column].fillna("").astype("string")

for column in band_columns:
    master_train[column] = master_train[column].fillna("UNKNOWN").astype("string")

for column in numerical_feature_columns:
    if column in master_train.columns:
        master_train[column] = master_train[column].replace([np.inf, -np.inf], np.nan).fillna(0.0)

if "learning_objective" in master_train.columns:
    master_train["learning_objective"] = master_train["learning_objective"].fillna("").astype("string")


# =========================================================
# 11. FINAL VALIDATION
# =========================================================
if len(master_train) != len(train_features):
    raise ValueError(f"Row count changed after merging. Expected {len(train_features):,}, found {len(master_train):,}.")

if master_train["response_id"].duplicated().any():
    raise ValueError("Duplicate response_id found in the final master dataset.")

if master_train["correct"].isna().any():
    raise ValueError("Missing labels remain in the final master dataset.")

if master_train["session_id"].isna().any():
    raise ValueError("Missing session_id found in the final master dataset.")


# =========================================================
# 12. ARRANGE IMPORTANT COLUMNS FIRST
# =========================================================
first_columns = ["response_id", "session_id", "learning_objective", "correct"]
first_columns = [column for column in first_columns if column in master_train.columns]

remaining_columns = [column for column in master_train.columns if column not in first_columns]
master_train = master_train[first_columns + remaining_columns]


# =========================================================
# 13. SAVE THE MODEL-READY MASTER TABLE
# =========================================================
master_train.to_parquet(MASTER_TRAIN_FILE, index=False, engine="pyarrow", compression="snappy")


# =========================================================
# 14. FINAL REPORT
# =========================================================
print("\nMaster training dataset created successfully.")
print("Train feature rows       :", f"{len(train_features):,}")
print("Train label rows         :", f"{len(train_labels):,}")
print("Transcript session rows  :", f"{len(transcript_features):,}")
print("Final master rows        :", f"{len(master_train):,}")
print("Final master columns     :", master_train.shape[1])
print("Missing transcript rows  :", f"{missing_transcript_rows:,}")
print("Positive labels          :", f"{int(master_train['correct'].sum()):,}")
print("Negative labels          :", f"{int((master_train['correct'] == 0).sum()):,}")
print("Saved file               :", MASTER_TRAIN_FILE)
print("File size                :", f"{MASTER_TRAIN_FILE.stat().st_size / 1024**2:.2f} MB")

display(master_train.head())

Label merge status:
_label_merge
both          35072
left_only         0
right_only        0
Name: count, dtype: int64

Transcript merge status:
_transcript_merge
both          35072
left_only         0
right_only        0
Name: count, dtype: int64

Master training dataset created successfully.
Train feature rows       : 35,072
Train label rows         : 35,072
Transcript session rows  : 22,821
Final master rows        : 35,072
Final master columns     : 36
Missing transcript rows  : 0
Positive labels          : 24,637
Negative labels          : 10,435
Saved file               : C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\03_master_dataset\master_train.parquet
File size                : 593.41 MB


,response_id,session_id,learning_objective,correct,learning_objective_id,source_file,transcript_text,student_text,tutor_text,total_turns,...,longest_tutor_streak_ratio,longest_student_streak_ratio,background_turn_ratio,session_turn_volume_band,session_duration_band,student_turn_share_band,student_response_length_band,student_short_turn_band,tutor_questioning_band,dialogue_switch_band
0,aaaavsh,bcaufvc,Knowing the value of each digit in numbers wit...,1,dqibnvd,bcaufvc.csv,"[BACKGROUND] [unclear]\n[TUTOR] Miss, I can't ...","Can you hear me? Hello?\nYeah, I hear you. Can...","Miss, I can't hear.\nCan you hear me? [unclear...",330,...,0.041420,0.026667,0.033333,HIGH,MEDIUM,MEDIUM,LOW,MEDIUM,LOW,HIGH
1,aaabhzi,eyutanf,Adding and subtracting tens to a 2-digit number.,1,eukmzxl,eyutanf.csv,[TUTOR] Yay! Hello!\n[STUDENT] Hello.\n[TUTOR]...,Hello.\nHello.\nI was one minute early.\nI'm o...,Yay! Hello!\nHello.\nHello. [Speaker:Backgroun...,280,...,0.020833,0.024390,0.046429,MEDIUM,HIGH,MEDIUM,HIGH,LOW,LOW,HIGH
2,aaahpnz,juptkxd,Comparing and ordering fractions by finding a ...,0,fjbqcsv,juptkxd.csv,[BACKGROUND] [unclear]\n[TUTOR] Is it me you a...,Hello.\nHello.\nYes.\nGood. How are you?\nNorm...,"Is it me you are looking for?\nOkay, so did yo...",291,...,0.021583,0.035971,0.044674,MEDIUM,HIGH,HIGH,HIGH,LOW,LOW,HIGH
3,aaajpom,ntwkcfj,Comparing fractions using reasoning.,0,acvbcev,ntwkcfj.csv,[BACKGROUND] [unclear]\n[TUTOR] Hello?\n[STUDE...,"Hello, Tobias. Can you hear me?\nOkay, that's ...","Hello?\nYes, I can hear you.\nGood. Why are yo...",265,...,0.022727,0.032258,0.033962,MEDIUM,HIGH,HIGH,HIGH,LOW,HIGH,MEDIUM
4,aaamwux,jqriibm,Counting in multiples.,0,krfuudx,jqriibm.csv,[BACKGROUND] [unclear]\n[TUTOR] Hello.\n[STUDE...,"Hello.\nGood. Okay, how was this week for you?...",Hello.\nHello.\nAre you feeling tired today?\n...,270,...,0.039216,0.019608,0.055556,MEDIUM,LOW,LOW,LOW,MEDIUM,MEDIUM,LOW
